# Лабораторная 10. Финальный debug case

Цель: разобрать плохой Spark job и объяснить, что именно не так.

In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import broadcast

spark = (SparkSession.builder.appName('lab-10-final-debug-case').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.driver.maxResultSize', '512m')
    .config('spark.sql.shuffle.partitions', '200')
    .config('spark.sql.adaptive.enabled', 'false')
    .config('spark.sql.autoBroadcastJoinThreshold', '-1')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base = Path('spark_core_data').absolute()
base_uri = base.as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
order_items = spark.read.parquet(f'{base_uri}/order_items')
products = spark.read.parquet(f'{base_uri}/products')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Плохой job
В коде намеренно есть проблемы: AQE выключен, 200 shuffle partitions для маленьких данных, broadcast отключён, несколько join, groupBy и `coalesce(1)` перед записью.

In [ ]:
bad = (
    orders
    .join(customers, 'customer_id')
    .join(order_items, 'order_id')
    .join(products, 'product_id')
    .groupBy('region', 'category', 'status')
    .agg(
        F.countDistinct('order_id').alias('orders_cnt'),
        F.sum(F.col('quantity') * F.col('item_price')).alias('revenue')
    )
    .orderBy(F.desc('revenue'))
)
bad.explain('formatted')

In [ ]:
out = base / 'final_bad_output'
if out.exists():
    shutil.rmtree(out)
bad.coalesce(1).write.mode('overwrite').parquet(out.as_uri())
out.as_uri()

## Диагностика
Откройте Spark UI и ответьте:

- Где появился лишний shuffle?
- Какой join strategy выбрал Spark?
- Что делает `coalesce(1)` перед записью?
- Почему 200 shuffle partitions плохо на этих данных?
- Какой stage самый дорогой? Почему?
- Где в плане видны `Exchange`, `SortMergeJoin`, `Sort`, `HashAggregate`?

## Улучшенный вариант
Запустите после диагностики. Важно не просто выполнить код, а объяснить каждое изменение.

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '16')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '20m')

better = (
    orders.select('order_id', 'customer_id', 'status')
    .join(broadcast(customers.select('customer_id', 'region')), 'customer_id')
    .join(order_items.select('order_id', 'product_id', 'quantity', 'item_price'), 'order_id')
    .join(broadcast(products.select('product_id', 'category')), 'product_id')
    .groupBy('region', 'category', 'status')
    .agg(
        F.countDistinct('order_id').alias('orders_cnt'),
        F.sum(F.col('quantity') * F.col('item_price')).alias('revenue')
    )
)
better.explain('formatted')

In [ ]:
out2 = base / 'final_better_output'
if out2.exists():
    shutil.rmtree(out2)
better.repartition(4).write.mode('overwrite').parquet(out2.as_uri())
out2.as_uri()

## Мини-отчёт

```text
Что было плохо:
...

Как я это увидел:
...

Что я изменил:
...

Почему это должно помочь:
...
```

Ожидаемые направления улучшения:

- включить AQE;
- уменьшить shuffle partitions для маленьких данных;
- broadcast маленькие справочники;
- заменить `coalesce(1)` на разумное количество partitions;
- выбирать только нужные колонки до join;
- объяснить каждое изменение через Spark UI и `explain`.